# Flight Delay Prediction Machine Learning Pipeline

This notebook demonstrates a complete, step-by-step Machine Learning pipeline for predicting flight delays using the 2015 U.S. Flight Delays dataset. The pipeline follows the industry-standard ML lifecycle:

**Pipeline Flowchart:**
`Real-World Problem` &rarr; `Data Collection` &rarr; `Dataset Understanding` &rarr; `Deep EDA` &rarr; `Data Cleaning` &rarr; `Missing Values` &rarr; `Outliers` &rarr; `Feature Engineering` &rarr; `Encoding` &rarr; `Scaling` &rarr; `X / y Split` &rarr; `Train/Test Split` &rarr; `Train Multiple Models` &rarr; `Evaluation` &rarr; `Model Comparison` &rarr; `Hyperparameter Tuning` &rarr; `Final Model Saving` &rarr; `New Unseen Data` &rarr; `Prediction`

## 1. Setup & Import Libraries
We import essential libraries for data manipulation, visualization, preprocessing, modeling, and evaluation.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add the parent/src directory to python path to use our modules
sys.path.append(os.path.abspath('../'))
from src.preprocessing import FlightPreprocessor
from src.models import ModelPipeline
from src.predict import predict_flight_delay

# Visual style settings
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12

print("Libraries successfully imported!")

## 2. Data Collection & Dataset Understanding
We load the datasets (`flights.csv`, `airlines.csv`, `airports.csv`) and inspect their structures, columns, and sizes.

In [ ]:
data_dir = '../dataset/First dataset/archive (2)'
preprocessor = FlightPreprocessor(sample_size=200000, random_state=42)

# Load raw datasets
raw_flights, airlines, airports = preprocessor.load_data(data_dir)

print("\n--- Airlines Sample ---")
display(airlines.head())

print("\n--- Airports Sample ---")
display(airports.head())

print("\n--- Flights Sample ---")
display(raw_flights.head(3))

## 3. Deep Exploratory Data Analysis (EDA)
Let's explore flight delays to reveal trends, distributions, correlation patterns, and peak delays by time of day, month, and airline.

In [ ]:
# Preprocess flights slightly for EDA purposes (cleaning + feature engineering)
df_clean = preprocessor.clean_data(raw_flights)
df_eda = preprocessor.engineer_features(df_clean)

# 1. Delay Distribution (Target variable count)
plt.figure(figsize=(6, 5))
sns.countplot(x='IS_DELAYED', data=df_eda, palette='Set2')
plt.title('Flight Delay Class Distribution (Target)')
plt.xlabel('Is Delayed (>= 15 Min)')
plt.ylabel('Count')
plt.xticks([0, 1], ['On-Time / Early (0)', 'Delayed (1)'])
plt.tight_layout()
plt.show()

delay_pct = df_eda['IS_DELAYED'].mean() * 100
print(f"Percentage of delayed flights: {delay_pct:.2f}%")
print(f"Percentage of on-time flights: {100 - delay_pct:.2f}%")

In [ ]:
# 2. Delays by Airline
plt.figure(figsize=(12, 6))
airline_delays = df_eda.groupby('AIRLINE_y')['IS_DELAYED'].mean().reset_index().sort_values(by='IS_DELAYED', ascending=False)
sns.barplot(x='IS_DELAYED', y='AIRLINE_y', data=airline_delays, palette='viridis')
plt.title('Flight Delay Rate by Airline')
plt.xlabel('Probability of Delay (>= 15 Min)')
plt.ylabel('Airline')
plt.tight_layout()
plt.show()

In [ ]:
# 3. Delays by Day of the Week
plt.figure(figsize=(10, 5))
day_delays = df_eda.groupby('DAY_OF_WEEK')['IS_DELAYED'].mean().reset_index()
sns.lineplot(x='DAY_OF_WEEK', y='IS_DELAYED', data=day_delays, marker='o', color='crimson', linewidth=2.5)
plt.title('Flight Delay Rate by Day of Week')
plt.xlabel('Day of the Week (1=Monday, 7=Sunday)')
plt.ylabel('Delay Rate')
plt.xticks(range(1, 8), ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])
plt.ylim(0, 0.3)
plt.show()

In [ ]:
# 4. Delays by Departure Hour
plt.figure(figsize=(12, 5))
hour_delays = df_eda.groupby('SCHEDULED_DEPARTURE_HOUR')['IS_DELAYED'].mean().reset_index()
sns.barplot(x='SCHEDULED_DEPARTURE_HOUR', y='IS_DELAYED', data=hour_delays, color='royalblue')
plt.title('Flight Delay Rate by Departure Hour (Time of Day)')
plt.xlabel('Scheduled Departure Hour')
plt.ylabel('Delay Rate')
plt.show()

In [ ]:
# 5. Correlation Heatmap of Numerical Features
plt.figure(figsize=(10, 8))
num_cols = ['MONTH', 'DAY', 'DAY_OF_WEEK', 'SCHEDULED_DEPARTURE_HOUR', 'SCHEDULED_ARRIVAL_HOUR', 
            'DISTANCE', 'SCHEDULED_TIME', 'FLIGHT_SPEED', 'ARRIVAL_DELAY', 'IS_DELAYED']
corr = df_eda[num_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', square=True, cbar_kws={'shrink': 0.8})
plt.title('Correlation Matrix of Numerical Features')
plt.tight_layout()
plt.show()

## 4. Preprocessing Pipeline
We run the entire cleaning, feature engineering, split, encoding, and scaling pipeline. 
The data split is **chronological**:
- **Train Set**: Months 1 - 10 (Jan - Oct)
- **Test Set**: Months 11 - 12 (Nov - Dec)

To prevent data leakage, categorical features (`AIRLINE`, `ORIGIN_AIRPORT`, `DESTINATION_AIRPORT`) are target-encoded or one-hot encoded using statistics calculated **exclusively on the training fold**.

In [ ]:
# Run preprocessor pipeline end-to-end
X_train, y_train, X_test, y_test, train_raw, test_raw = preprocessor.run_pipeline(data_dir)

print("\n--- Processed Feature Matrix Shapes ---")
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

print("\n--- First 3 Rows of Features ---")
display(X_train.head(3))

## 5. Model Training & Comparison
We train multiple models with different architectural approaches and evaluate their classification performance:
1. **Logistic Regression** (baseline linear classifier)
2. **Decision Tree** (non-linear baseline)
3. **Random Forest** (bagging ensemble method)
4. **HistGradientBoosting** (boosting ensemble method - scikit-learn's optimized LightGBM alternative)

In [ ]:
pipeline = ModelPipeline(random_state=42)

# Train all models and plot ROC curves
comparison_df = pipeline.train_and_evaluate_all(X_train, y_train, X_test, y_test, reports_dir='../reports/figures')

print("\n--- Performance Comparison Table ---")
display(comparison_df)

In [ ]:
# Plot Confusion Matrices for all models
pipeline.plot_confusion_matrices(X_test, y_test, reports_dir='../reports/figures')

## 6. Hyperparameter Tuning
Let's select the best-performing model (based on ROC-AUC) and optimize its hyperparameters using `RandomizedSearchCV` to maximize its predictive performance.

In [ ]:
best_model_name = comparison_df['ROC-AUC'].idxmax()
print(f"Best Model Selected for Tuning: {best_model_name}")

tuned_model = pipeline.tune_best_model(X_train, y_train, X_test, y_test, best_model_name)

# Save tuned model and preprocessor
pipeline.save_pipeline(tuned_model, preprocessor, models_dir='../models')

## 7. New Unseen Data & Predictions
We load the finalized pipeline and demonstrate delay prediction for a newly scheduled flight.

In [ ]:
# Example of a newly scheduled flight (unseen data)
new_flight = {
    'YEAR': 2015,
    'MONTH': 12,
    'DAY': 24, # Christmas Eve
    'DAY_OF_WEEK': 4, # Thursday
    'AIRLINE': 'UA',
    'ORIGIN_AIRPORT': 'SFO',
    'DESTINATION_AIRPORT': 'JFK',
    'SCHEDULED_DEPARTURE': 1730, # 5:30 PM
    'SCHEDULED_ARRIVAL': 2145, # 9:45 PM
    'SCHEDULED_TIME': 255.0,
    'DISTANCE': 2586.0,
    'CANCELLED': 0,
    'DIVERTED': 0
}

print("Predicting delay for scheduled flight (SFO -> JFK on UA):")
results = predict_flight_delay(new_flight, models_dir='../models')
display(results)